# Notebook 22c — Wu 2003: S-B3 eta_col-widened top-up data generation

**This notebook has not been executed. Run it under `caffeinate` like notebook 22b, and expect it to
be slower per row than notebook 22b's mixed-wide branch** -- widening a *third* dimension
(`eta_col`, on top of `alpha`/`beta_r`) to uniform increases how often multiple parameters land
simultaneously near their extreme bounds at once, which is exactly what made ODE convergence slow in
every prior-widening experiment this project has tried (see `cstr_sbi.recycle.priors.MixedWidePrior`
docstring and notebook 22b's own history). Target sizes here are deliberately smaller than notebook
22b's (2000/60/150 vs. 5000/120/400) to bound the cost of this experiment.

## Why this bank, and why it's a top-up, not a replacement

Notebook 31b's assessment found that `eta_col` (kept narrow lognormal in notebook 22b's mixed-wide
prior) still fails SBC (KS p=0.0001) and is the direct cause of one real classification error
(`W12_snowball_compound`, true=`multi` predicted as `reactor` because the posterior's `eta_col`
estimate shrinks from the true 0.80 up to 0.867, just past the 0.85 classifier threshold). `xi_reb`
also fails SBC (p=0.008) but its point estimates are already the *best* of the four parameters
(MAE=0.020), so it is deliberately left narrow here -- only `eta_col` is widened.

This notebook produces a **top-up** bank sampled from `mixed_wide_prior_4d_sb3(uniform_idx=(0,1,2))`
(`alpha, beta_r, eta_col` uniform; `xi_reb` still truncated lognormal). It is meant to be **pooled**
with notebook 22b's existing 5000-row mixed-wide bank (not replace it) via notebook 22d's aggregation
step and notebook 24c's pooled-multiproposal training -- this reuses the ~14.5 hours already spent
generating notebook 22b's bank instead of discarding it, following the same "pool historical banks
each tagged with their own true generating prior" pattern already validated elsewhere in this project
(`SNPE`'s atomic proposal-correction machinery, see notebook 24c for the exact mechanism).

## Contents

1. Setup
2. Generation procedure
3. Generate top-up banks (train / SBC reduced / SBC confirm)
4. Dataset shapes, NaN checks
5. Diagnostics for suspicious generated samples
6. Save / load round-trip verification and acceptance criteria

In [ ]:
import pathlib, sys, time, importlib
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

cwd = pathlib.Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(ROOT / 'src'))

import cstr_sbi.recycle.sb3 as sb3
sb3 = importlib.reload(sb3)
from cstr_sbi.recycle.sb3 import (
    PRIOR_HIGH_SB3_4D,
    PRIOR_LOW_SB3_4D,
    SB3_PARAM_NAMES,
    mixed_wide_prior_4d_sb3,
    deterministic_window_sb3,
    noisy_replicates_sb3,
    nominal_warm_start_sb3,
)
from cstr_sbi.recycle.summaries import N_SUMMARIES_SB, compute_summaries

DATA = ROOT / 'data'; DATA.mkdir(exist_ok=True)
RESULTS = ROOT / 'results'; RESULTS.mkdir(exist_ok=True)
FIGURES = ROOT / 'figures'; FIGURES.mkdir(exist_ok=True)
LOGS = ROOT / 'logs'; LOGS.mkdir(exist_ok=True)

NOISE_PCT = 0.003
T_FINAL_H = 4.0
N_SAVE = 240

# Smaller than notebook 22b's mixed-wide branch (5000/120/400): this is a top-up bank, generation
# is expected to be slower per row than notebook 22b's 2-dim-wide prior, and the goal is only to
# supplement eta_col's coverage, not to replace the existing bank.
N_TOPUP_TRAIN = 2000
N_TOPUP_SBC_REDUCED = 60
N_TOPUP_SBC_CONFIRM = 150

UNIFORM_IDX_TOPUP = (0, 1, 2)  # alpha, beta_r, eta_col uniform; xi_reb stays narrow lognormal

CHECKPOINT_EVERY = 20
TIME_CHECKPOINT_S = 60
PROGRESS_EVERY_ATTEMPTS = 5
STEADY_MAX_REL_DRIFT = 0.05
MAX_NORM_CHANNEL = 3.0
MAX_ABS_RAW_OBS = 3.0e7
PROGRESS_LOG_PATH = LOGS / 'notebook22c_progress.log'

ETAWIDE_TRAIN_PATH = DATA / f'wu2003_sbi_train_sb3_etawide_topup_n{N_TOPUP_TRAIN}.npz'
ETAWIDE_SBC_R_PATH = RESULTS / f'24c_sysII_sb3_etawide_topup_sbc_set_n{N_TOPUP_SBC_REDUCED}.npz'
ETAWIDE_SBC_C_PATH = RESULTS / f'24c_sysII_sb3_etawide_topup_sbc_set_n{N_TOPUP_SBC_CONFIRM}.npz'
ETAWIDE_THETA_FIG_PATH = FIGURES / 'nb22c_etawide_topup_theta_ranges.png'
ETAWIDE_DIAG_FIG_PATH = FIGURES / 'nb22c_etawide_topup_summary_norm_diagnostics.png'

y0_sb3 = nominal_warm_start_sb3()
OI = ['#000000', '#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7']
print(f'N_TOPUP_TRAIN={N_TOPUP_TRAIN}, N_TOPUP_SBC_REDUCED={N_TOPUP_SBC_REDUCED}, N_TOPUP_SBC_CONFIRM={N_TOPUP_SBC_CONFIRM}')
print(f'UNIFORM_IDX_TOPUP={UNIFORM_IDX_TOPUP} (alpha, beta_r, eta_col uniform; xi_reb lognormal)')
print(f'ETAWIDE_TRAIN_PATH={ETAWIDE_TRAIN_PATH.relative_to(ROOT)}')

## Section 1 — Generation procedure

Mirrors notebook 22b's mixed-wide-branch generator exactly (same checkpointing every
`CHECKPOINT_EVERY` accepted rows or `TIME_CHECKPOINT_S` seconds, same ETA reporting, same
`_check_bank_horizon` guard against silently reusing a bank generated under different settings),
just pointed at the new 3-dim-wide prior and smaller target sizes.

In [ ]:
def simulate_summary_sb3(theta4, rng, noise_pct=NOISE_PCT, t_final_h=T_FINAL_H, n_save=N_SAVE):
    theta4 = np.asarray(theta4, dtype=np.float32)
    t_h, raw_obs = deterministic_window_sb3(theta4, y0=y0_sb3, t_final_h=t_final_h, n_save=n_save)
    raw_obs = np.asarray(raw_obs)
    noisy = noisy_replicates_sb3(raw_obs, n_replicates=1, rng=rng, noise_pct=noise_pct)[0]
    summary = compute_summaries(noisy, 'S-B', np.asarray(t_h))
    return np.asarray(t_h), raw_obs, noisy, summary


def window_diag(raw_obs):
    raw_obs = np.asarray(raw_obs)
    check_idx = [0, 1, 2, 5, 6, 7, 8, 9, 10]
    norm_idx = [7, 8, 9, 10]
    q_start = max(0, 3 * len(raw_obs) // 4)
    last_q = raw_obs[q_start:]
    final = raw_obs[-1]
    q_mean = last_q.mean(axis=0)
    denom = np.maximum(np.abs(q_mean), 1e-6)
    rel_drift = np.abs(final - q_mean) / denom
    return {
        'max_rel_drift_sb': float(np.max(rel_drift[check_idx])),
        'mean_rel_drift_sb': float(np.mean(rel_drift[check_idx])),
        'max_norm_channel': float(np.max(np.abs(raw_obs[:, norm_idx]))),
        'max_abs_raw': float(np.max(np.abs(raw_obs))),
    }


def is_acceptable_window(diag):
    return (
        diag['max_rel_drift_sb'] <= STEADY_MAX_REL_DRIFT
        and diag['max_norm_channel'] <= MAX_NORM_CHANNEL
        and diag['max_abs_raw'] <= MAX_ABS_RAW_OBS
    )


def _check_bank_horizon(data, candidate_name):
    stored_t_final_h = float(data['t_final_h']) if 't_final_h' in data.files else None
    stored_n_save = int(data['n_save']) if 'n_save' in data.files else None
    mismatch = (
        (stored_t_final_h is not None and not np.isclose(stored_t_final_h, T_FINAL_H))
        or (stored_n_save is not None and stored_n_save != N_SAVE)
    )
    if mismatch:
        raise ValueError(
            f'{candidate_name} was generated with t_final_h={stored_t_final_h}, n_save={stored_n_save}, '
            f'which does not match the current T_FINAL_H={T_FINAL_H}, N_SAVE={N_SAVE}.'
        )


def load_matching_bank(out_path, label, emit_progress):
    if out_path.exists():
        data = np.load(out_path)
        _check_bank_horizon(data, out_path.name)
        emit_progress(f'[{label}] loading existing final bank: {out_path.name}')
        return (
            list(np.asarray(data['thetas4'], dtype=np.float32)),
            list(np.asarray(data['summaries'], dtype=np.float32)),
            int(data['attempts']) if 'attempts' in data.files else len(data['thetas4']),
            int(data['reject_nonfinite']) if 'reject_nonfinite' in data.files else 0,
            int(data['reject_diag']) if 'reject_diag' in data.files else 0,
        )
    partial_path = out_path.with_suffix('.partial.npz')
    if partial_path.exists():
        data = np.load(partial_path)
        _check_bank_horizon(data, partial_path.name)
        theta_rows = list(np.asarray(data['thetas4'], dtype=np.float32))
        summary_rows = list(np.asarray(data['summaries'], dtype=np.float32))
        attempts = int(data['attempts']) if 'attempts' in data.files else len(theta_rows)
        reject_nonfinite = int(data['reject_nonfinite']) if 'reject_nonfinite' in data.files else 0
        reject_diag = int(data['reject_diag']) if 'reject_diag' in data.files else 0
        emit_progress(
            f'[{label}] resuming from partial: accepted={len(theta_rows)}/{out_path.name}, '
            f'attempts={attempts}, reject_nonfinite={reject_nonfinite}, reject_diag={reject_diag}'
        )
        return theta_rows, summary_rows, attempts, reject_nonfinite, reject_diag
    emit_progress(f'[{label}] starting from scratch for {out_path.name}')
    return [], [], 0, 0, 0


def save_bank(out_path, theta_rows, summary_rows, seed, attempts, reject_nonfinite, reject_diag, note):
    theta_arr = np.asarray(theta_rows, dtype=np.float32)
    summary_arr = np.asarray(summary_rows, dtype=np.float32)
    assert summary_arr.shape[1] == N_SUMMARIES_SB
    np.savez(
        out_path, thetas4=theta_arr, summaries=summary_arr, seed=seed, structure='S-B3',
        t_final_h=T_FINAL_H, n_save=N_SAVE, attempts=attempts, reject_nonfinite=reject_nonfinite,
        reject_diag=reject_diag, note=note, prior_kind='etawide_topup',
        uniform_idx=np.asarray(UNIFORM_IDX_TOPUP, dtype=np.int64),
    )
    return theta_arr, summary_arr


def generate_etawide_topup_bank(out_path, n_rows, seed, label, prior_etawide,
                                checkpoint_every=CHECKPOINT_EVERY, time_checkpoint_s=TIME_CHECKPOINT_S,
                                progress_every_attempts=PROGRESS_EVERY_ATTEMPTS):
    def emit_progress(message, reset=False):
        print(message, flush=True)
        mode = 'w' if reset else 'a'
        with PROGRESS_LOG_PATH.open(mode, encoding='utf-8') as fh:
            fh.write(message + '\n')

    theta_rows, summary_rows, attempts, reject_nonfinite, reject_diag = load_matching_bank(out_path, label, emit_progress)
    if len(theta_rows) >= n_rows:
        theta_arr = np.asarray(theta_rows[:n_rows], dtype=np.float32)
        summary_arr = np.asarray(summary_rows[:n_rows], dtype=np.float32)
        emit_progress(f'[{label}] existing bank already satisfies target rows: {theta_arr.shape[0]}/{n_rows}')
        return theta_arr, summary_arr

    partial_path = out_path.with_suffix('.partial.npz')
    emit_progress(f'[{label}] target={out_path.name} partial={partial_path.name} n_rows={n_rows} seed={seed}')
    rng = np.random.default_rng(seed)
    t0 = time.time()
    last_checkpoint_t = t0

    def report_progress(tag):
        elapsed = time.time() - t0
        accepted = len(theta_rows)
        accept_frac = accepted / max(attempts, 1)
        accepted_per_min = accepted / max(elapsed / 60.0, 1e-9)
        remaining = max(n_rows - accepted, 0)
        eta_min = remaining / max(accepted_per_min, 1e-9) if accepted > 0 else float('nan')
        emit_progress(
            f'[{label}] {tag}: accepted={accepted}/{n_rows}, attempts={attempts}, '
            f'accept_frac={accept_frac:.3f}, accepted_per_min={accepted_per_min:.2f}, '
            f'reject_nonfinite={reject_nonfinite}, reject_diag={reject_diag}, '
            f'elapsed_min={elapsed / 60.0:.1f}, ETA_remaining_min={eta_min:.1f}'
        )

    def checkpoint_now():
        np.savez(partial_path, thetas4=np.asarray(theta_rows, dtype=np.float32),
                 summaries=np.asarray(summary_rows, dtype=np.float32), attempts=attempts,
                 reject_nonfinite=reject_nonfinite, reject_diag=reject_diag,
                 t_final_h=T_FINAL_H, n_save=N_SAVE)
        report_progress('checkpoint')

    while len(theta_rows) < n_rows:
        attempts += 1
        theta4 = prior_etawide.sample((1,)).numpy()[0]
        _, raw_obs, noisy, summary = simulate_summary_sb3(theta4, rng)
        if not np.isfinite(raw_obs).all() or not np.isfinite(noisy).all() or not np.isfinite(summary).all():
            reject_nonfinite += 1
            if attempts % progress_every_attempts == 0:
                report_progress('heartbeat')
            continue
        diag = window_diag(raw_obs)
        if not is_acceptable_window(diag):
            reject_diag += 1
            if attempts % progress_every_attempts == 0:
                report_progress('heartbeat')
            continue
        theta_rows.append(np.asarray(theta4, dtype=np.float32))
        summary_rows.append(np.asarray(summary, dtype=np.float32))
        if attempts % progress_every_attempts == 0:
            report_progress('heartbeat')
        due_by_count = len(theta_rows) % checkpoint_every == 0 or len(theta_rows) == n_rows
        due_by_time = (time.time() - last_checkpoint_t) >= time_checkpoint_s
        if due_by_count or due_by_time:
            checkpoint_now()
            last_checkpoint_t = time.time()

    theta_arr, summary_arr = save_bank(
        out_path, theta_rows, summary_rows, seed, attempts, reject_nonfinite, reject_diag,
        'Eta_col-widened top-up S-B3 summary bank (alpha, beta_r, eta_col uniform; xi_reb lognormal)',
    )
    emit_progress(f'[{label}] saved {out_path.name}: theta4={theta_arr.shape}, summaries={summary_arr.shape}')
    return theta_arr, summary_arr

## Section 2 — Generate top-up banks (train / SBC reduced / SBC confirm)

**Run this under `caffeinate -i`, e.g.:**
```
caffeinate -i jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 \
    notebooks/22c_wu2003_data_generation_etawide_sb3.ipynb
```
Expect this to be slower per accepted row than notebook 22b's 2-dim-wide branch (which took ~14.5h
for 5520 rows) -- track progress via `logs/notebook22c_progress.log`.

In [ ]:
prior_etawide = mixed_wide_prior_4d_sb3(uniform_idx=UNIFORM_IDX_TOPUP)

theta_train_etawide, summary_train_etawide = generate_etawide_topup_bank(
    ETAWIDE_TRAIN_PATH, N_TOPUP_TRAIN, seed=222010, label='etawide_topup_train', prior_etawide=prior_etawide,
)
theta_sbc_r_etawide, summary_sbc_r_etawide = generate_etawide_topup_bank(
    ETAWIDE_SBC_R_PATH, N_TOPUP_SBC_REDUCED, seed=222020, label='etawide_topup_sbc_reduced', prior_etawide=prior_etawide,
)
theta_sbc_c_etawide, summary_sbc_c_etawide = generate_etawide_topup_bank(
    ETAWIDE_SBC_C_PATH, N_TOPUP_SBC_CONFIRM, seed=222030, label='etawide_topup_sbc_confirm', prior_etawide=prior_etawide,
)

## Section 3 — Dataset shapes, NaN checks

In [ ]:
print('=== Eta_col-widened top-up bank shapes ===')
print('  theta_train_etawide   :', theta_train_etawide.shape)
print('  summary_train_etawide :', summary_train_etawide.shape)
print('  theta_sbc_r_etawide   :', theta_sbc_r_etawide.shape)
print('  theta_sbc_c_etawide   :', theta_sbc_c_etawide.shape)
print()
print('=== NaN / finite checks ===')
print('  train summaries finite   :', bool(np.isfinite(summary_train_etawide).all()))
print('  reduced SBC finite       :', bool(np.isfinite(summary_sbc_r_etawide).all()))
print('  confirm SBC finite       :', bool(np.isfinite(summary_sbc_c_etawide).all()))
print()
print('=== Parameter ranges against the S-B3 box ===')
for idx, name in enumerate(SB3_PARAM_NAMES):
    kind = 'uniform' if idx in UNIFORM_IDX_TOPUP else 'lognormal'
    print(f'  {name:8s} ({kind:9s}): train [{theta_train_etawide[:, idx].min():.3f}, {theta_train_etawide[:, idx].max():.3f}] '
          f'within [{PRIOR_LOW_SB3_4D[idx]:.3f}, {PRIOR_HIGH_SB3_4D[idx]:.3f}]')

fig, axes = plt.subplots(2, 2, figsize=(10, 6), constrained_layout=True)
for ax, idx, name in zip(axes.ravel(), range(4), SB3_PARAM_NAMES):
    ax.hist(theta_train_etawide[:, idx], bins=30, color=OI[2], alpha=0.8, edgecolor='k', linewidth=0.4)
    ax.axvline(PRIOR_LOW_SB3_4D[idx], color=OI[6], ls='--', lw=1.0)
    ax.axvline(PRIOR_HIGH_SB3_4D[idx], color=OI[6], ls='--', lw=1.0)
    ax.set_title(name)
    ax.grid(True, alpha=0.2)
plt.savefig(ETAWIDE_THETA_FIG_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved {ETAWIDE_THETA_FIG_PATH.name}')

## Section 4 — Diagnostics for suspicious generated samples

In [ ]:
def robust_flag_summary_rows(summary_arr, z_thresh=6.0):
    norms = np.linalg.norm(summary_arr, axis=1)
    med = float(np.median(norms))
    mad = float(np.median(np.abs(norms - med)))
    z = np.zeros_like(norms) if mad < 1e-12 else 0.6745 * (norms - med) / mad
    flagged = np.where(np.abs(z) > z_thresh)[0]
    return norms, z, flagged


def diagnose_bank(summary_arr, label):
    norms, robust_z, flagged = robust_flag_summary_rows(summary_arr)
    print(f'=== Diagnostics: {label} ===')
    print(f'  rows={len(summary_arr)}  flagged_by_summary_norm={len(flagged)}')
    print(f'  summary_norm median={np.median(norms):.3f}  p99={np.percentile(norms, 99):.3f}  max={np.max(norms):.3f}')
    return {'norms': norms, 'robust_z': robust_z, 'flagged': flagged}

train_diag = diagnose_bank(summary_train_etawide, 'etawide_topup_train')
sbc_r_diag = diagnose_bank(summary_sbc_r_etawide, 'etawide_topup_sbc_reduced')
sbc_c_diag = diagnose_bank(summary_sbc_c_etawide, 'etawide_topup_sbc_confirm')

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
for ax, title, diag in zip(axes, ['train', 'sbc_reduced', 'sbc_confirm'], [train_diag, sbc_r_diag, sbc_c_diag]):
    ax.hist(diag['norms'], bins=30, color=OI[5], alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('summary L2 norm')
    ax.set_ylabel('count')
plt.savefig(ETAWIDE_DIAG_FIG_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved {ETAWIDE_DIAG_FIG_PATH.name}')

## Section 5 — Save / load round-trip verification and acceptance criteria

This branch passes if:
1. all three datasets are generated with the requested row counts,
2. all summary vectors are finite and have the expected `S-B` dimension,
3. `alpha`, `beta_r`, `eta_col` samples span their full uniform range; `xi_reb` stays within its
   narrower lognormal-implied range,
4. no bank has more than 1% robust summary-norm outliers on the diagnostics pass.

In [ ]:
train_rt = np.load(ETAWIDE_TRAIN_PATH)
sbc_r_rt = np.load(ETAWIDE_SBC_R_PATH)
sbc_c_rt = np.load(ETAWIDE_SBC_C_PATH)
print('=== Round-trip verification ===')
print('  train   :', train_rt['thetas4'].shape, train_rt['summaries'].shape)
print('  sbc_r   :', sbc_r_rt['thetas4'].shape, sbc_r_rt['summaries'].shape)
print('  sbc_c   :', sbc_c_rt['thetas4'].shape, sbc_c_rt['summaries'].shape)

within_support = bool(
    (theta_train_etawide >= PRIOR_LOW_SB3_4D).all() and (theta_train_etawide <= PRIOR_HIGH_SB3_4D).all()
)

criteria = [
    ('training row count', theta_train_etawide.shape[0] == N_TOPUP_TRAIN, f'{theta_train_etawide.shape[0]} vs {N_TOPUP_TRAIN}'),
    ('reduced SBC row count', theta_sbc_r_etawide.shape[0] == N_TOPUP_SBC_REDUCED, f'{theta_sbc_r_etawide.shape[0]} vs {N_TOPUP_SBC_REDUCED}'),
    ('confirm SBC row count', theta_sbc_c_etawide.shape[0] == N_TOPUP_SBC_CONFIRM, f'{theta_sbc_c_etawide.shape[0]} vs {N_TOPUP_SBC_CONFIRM}'),
    ('all summaries finite', bool(np.isfinite(np.vstack([summary_train_etawide, summary_sbc_r_etawide, summary_sbc_c_etawide])).all()), 'finite check'),
    ('summary dimension is S-B', summary_train_etawide.shape[1] == N_SUMMARIES_SB, str(summary_train_etawide.shape[1])),
    ('samples remain within strict support', within_support, 'support bounds check'),
    ('nonzero spread in every parameter', bool((theta_train_etawide.std(axis=0) > 0).all()), np.round(theta_train_etawide.std(axis=0), 4).tolist()),
    ('train summary-norm outliers <= 1%', len(train_diag['flagged']) <= max(1, int(0.01 * len(theta_train_etawide))), f"{len(train_diag['flagged'])}/{len(theta_train_etawide)}"),
]
for name, passed, value in criteria:
    print(f"{'PASS' if passed else 'FAIL'} | {name}: {value}")

TOPUP_BRANCH_COMPLETE = all(passed for _, passed, _ in criteria)
print(f'TOPUP_BRANCH_COMPLETE={TOPUP_BRANCH_COMPLETE}')
print()
print('Next step: notebook 22d aggregates this top-up bank with notebook 22b\'s existing mixed-wide bank.')